# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset may include several `RecordSet` entities, each with its own `@id`. We'll list them here, along with their included fields.

In [ ]:
# List all record sets and their fields with @id
print("\nAvailable record sets and their fields:\n" + "-"*40)
record_sets = []
for rs in metadata.record_sets:
    print(f"Record set: {rs.name} (id: {rs.id})")
    record_sets.append(rs.id)
    field_lines = []
    for fld in rs.fields:
        dtype = getattr(fld, 'data_type', '')
        field_lines.append(f"  - {fld.name} (@id: {fld.id}) "+(f"[type: {dtype}]" if dtype else ""))
    print("\n".join(field_lines))
    print("")
if not record_sets:
    print("No record sets found in the Croissant metadata. The dataset may be purely file-based or the schema uses specialized structures.")
# For exploration: print some sample records if any record set is found
if record_sets:
    print(f"Sample record from first RecordSet ({record_sets[0]}):")
    sample_recordset = record_sets[0]
    record_generator = dataset.records(record_set=sample_recordset)
    for i, record in enumerate(record_generator):
        print(record)
        if i > 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract data from all available record sets into pandas DataFrames.

In [ ]:
# Extract data from each record set
# The record_sets list was populated in the previous cell
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set} with shape {dataframes[record_set].shape}")
        print(dataframes[record_set].columns.tolist())
        display(dataframes[record_set].head())
    else:
        print(f"No records found for record set {record_set}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll run some basic EDA if there is at least one DataFrame with numeric data
if dataframes:
    # Pick the first record set for demo
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    
    # Try to find a numeric field for demonstration (e.g. coverage, peptide count, MW, pI, etc.)
    # We'll guess by looking for columns with float or int dtype
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_columns:
        print("No numeric columns available for EDA in record set.")
    else:
        numeric_field = numeric_columns[0]
        print(f"Using numeric field: {numeric_field}")

        # Remove potential outliers (upper/lower 1st percentile)
        lower = df[numeric_field].quantile(0.01)
        upper = df[numeric_field].quantile(0.99)
        filtered_df = df[(df[numeric_field] > lower) & (df[numeric_field] < upper)].copy()
        print(f"Filtered records with {numeric_field} in 1st-99th percentile: {filtered_df.shape[0]} records left.")
        display(filtered_df[[numeric_field]].head())

        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized values for {numeric_field}:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Try grouping by another (probably categorical/text/id) field
        # We'll pick the first non-numeric column
        candidate_group_fields = [col for col in df.columns if col not in numeric_columns]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            print(f"Grouping records by {group_field} (showing mean of numeric field):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot distributions of numeric fields, or scatterplots for two selected fields.

In [ ]:
if dataframes:
    df = dataframes[record_set_id]
    # If there's more than one numeric field, plot their relationship
    if len(numeric_columns) > 1:
        plt.figure(figsize=(6,4))
        plt.scatter(df[numeric_columns[0]], df[numeric_columns[1]], alpha=0.5)
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.title(f'Scatter plot: {numeric_columns[0]} vs {numeric_columns[1]}')
        plt.show()
    else:
        # Histogram of the numeric field
        plt.figure(figsize=(6, 4))
        df[numeric_field].hist(bins=30)
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.title(f'Distribution of {numeric_field}')
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² Croissant dataset for mass spectrometry analyses of extracellular vesicles from human mast cells. We loaded Croissant metadata, listed all record sets and their fields, extracted records into DataFrames, performed basic EDA including outlier removal and normalization, and visualized distributions of selected numeric fields. This workflow demonstrates how to get started with FAIR datasets using `mlcroissant` and can be extended for detailed downstream bioinformatics or ML analyses.